# Linear RL-FMU stability benchmark

This notebook compares the same DC series RL circuit in native DPsim, FMI Model Exchange (ME), and FMI Co-Simulation (CS). It summarizes the fixed-step results and the explicit-CS stability sweep. The recorded traces and plot are generated by `run_benchmark.py` under `build-fmi/linear_rl_stability/results/`.

## Explicit benchmark settings

| Setting | Value | Applied to |
|---|---:|---|
| DC source | \(V_s=10\,\mathrm{V}\), step applied at \(t=0\) | Native, ME, CS |
| Native grid resistance | \(R_g=1\,\Omega\) | Native, ME, CS |
| FMU resistance | \(R_f=0.1\,\Omega\) | Native Rf component and both FMUs |
| FMU inductance | \(L=20\,\mu\mathrm{H}\) | Native inductor and both FMUs |
| Total series resistance | \(R_g+R_f=1.1\,\Omega\) | All three |
| Electrical time constant | \(L/(R_g+R_f)=18.1818\,\mu\mathrm{s}\) | All three |
| Initial inductor current | \(i(0)=0\,\mathrm{A}\) | Native, ME state, CS state |
| Initial terminal voltage / FMU voltage input | \(0\,\mathrm{V}\) before first network solve | Native, ME, CS |
| Steady-state initialization | Disabled; no load-flow initialization | Native, ME, CS |
| Network timestep | \(50\,\mu\mathrm{s}\) nominal | Native, ME, CS |
| CS communication step | Equal to network step; \(50\,\mu\mathrm{s}\) nominal | CS |
| Stop time | \(5\,\mathrm{ms}\) | Every sweep run |
| CS timestep sweep | 20, 30, 35, 38, 39, 40, 40.13, 40.25, 41, 45, 50 \(\mu\mathrm{s}\) | All three variants |
| FMI version and port | FMI 2.0; voltage input `u` (V), passive current output `i` (A) | ME and CS |
| CS coupling | One `doStep()` per network step; PCC voltage held over the step; last completed FMU current used by MNA | CS |

All three variants use the same zero-current start and the same source step. No variant is initialized at its nonzero steady-state current. The Simulink source model is retained, but this host lacks Simulink Coder; the executed FMI 2.0 ME and CS FMUs were built from the reproducible C++ fallback exporter. The CS implementation integrates the held-input RL equation exactly within each `doStep`.

## Analytical response and CS stability boundary

For the physical circuit,

\[L\frac{di}{dt}=V_s-(R_g+R_f)i,\qquad i(t)=\frac{V_s}{R_g+R_f}\left(1-e^{-(R_g+R_f)t/L}\right).\]

In explicit CS, the MNA network solves with the previous FMU current, and the FMU holds the resulting PCC voltage during the next communication interval. With an exact internal RL step, the feedback multiplier is

\[q(h)=\left(1+\frac{R_g}{R_f}\right)e^{-R_fh/L}-\frac{R_g}{R_f}.\]

Stability requires \(|q|<1\). The first boundary is at \(q=-1\):

\[h_{\rm crit}=\frac{L}{R_f}\ln\left(\frac{R_g+R_f}{R_g-R_f}\right)=40.134139\,\mu\mathrm{s}.\]

This boundary describes the selected single-pass explicit CS coupling. It does not imply that every FMI CS master or coupling method is unstable.

In [ ]:
from pathlib import Path
import json
import math
import pandas as pd
from IPython.display import display, Image, Markdown

# Find the repository root whether this notebook is opened from the repo root
# or from its example directory.
candidates = [Path.cwd(), *Path.cwd().parents]
repo_root = next(
    p for p in candidates if (p / "dpsim-fmi/examples/linear_rl_stability").is_dir()
)
results_dir = repo_root / "build-fmi/linear_rl_stability/results"
summary = pd.read_csv(results_dir / "summary.csv")
summary.columns = summary.columns.str.strip()
metrics = json.loads((results_dir / "metrics.json").read_text())

VS, RG, RF, L = 10.0, 1.0, 0.1, 20e-6
hcrit = L / RF * math.log((RG + RF) / (RG - RF))
print(f"Analytical explicit-CS boundary: {hcrit * 1e6:.6f} µs")
print(f"Recorded result directory: {results_dir}")

## Measured results at the nominal 50 µs step

In [ ]:
nominal = summary[summary["step_us"] == 50.0].copy()
nominal = nominal[
    [
        "mode",
        "step_us",
        "peak_current_A",
        "final_current_A",
        "max_abs_error_A",
        "rms_error_A",
    ]
]
nominal = nominal.rename(
    columns={
        "mode": "Variant",
        "step_us": "Step (µs)",
        "peak_current_A": "Peak |i| (A)",
        "final_current_A": "Final i (A)",
        "max_abs_error_A": "Max |error| vs analytic (A)",
        "rms_error_A": "RMS error (A)",
    }
)
display(nominal.round(6))


# Independently check the recorded initial current and native/ME trace match.
def load_trace(mode, step_us):
    trace = pd.read_csv(results_dir / "raw" / f"{mode}_{step_us:g}us.csv")
    trace.columns = trace.columns.str.strip()
    return trace


traces = {mode: load_trace(mode, 50.0) for mode in ("native", "me", "cs")}
for mode, trace in traces.items():
    assert (
        abs(float(trace.loc[0, "i_A"])) < 1e-12
    ), f"{mode} initial current is not zero"
    assert (
        abs(float(trace.loc[0, "u_V"])) < 1e-12
    ), f"{mode} initial terminal voltage is not zero"
assert (traces["native"]["i_A"] - traces["me"]["i_A"]).abs().max() < 1e-12
print(
    "Shared initial conditions verified: current 0 A and terminal voltage 0 V in all three traces."
)
print("Native and ME nominal current traces match to CSV precision.")

## Sweep around the predicted boundary

In [ ]:
boundary_steps = [40.0, 40.13, 40.25, 41.0, 45.0, 50.0]
cs = summary[(summary["mode"] == "cs") & summary["step_us"].isin(boundary_steps)].copy()
cs["Classification"] = cs["cs_stable_by_analysis"].map(
    {True: "bounded by recurrence", False: "unstable by recurrence"}
)
cs = cs[
    ["step_us", "cs_multiplier", "peak_current_A", "final_current_A", "Classification"]
]
cs = cs.rename(
    columns={
        "step_us": "CS step (µs)",
        "cs_multiplier": "Multiplier q",
        "peak_current_A": "Measured peak |i| (A)",
        "final_current_A": "Final i (A)",
    }
)
display(cs.round(6))

## Plots and interpretation

The left panel separates the bounded native/ME traces from the rapidly growing CS response so both remain legible. The bottom panel compares measured CS peak current with the analytical boundary.

In [ ]:
display(Image(filename=str(results_dir / "linear_rl_stability.png")))

At 50 µs, native and ME converge to 9.090909 A and are identical at the recorded precision. The explicit-CS multiplier is \(q=-1.433191\), so each communication step increases the alternating error; the recorded peak reaches \(2.71\times10^{16}\,\mathrm{A}\) over 5 ms. At 40.00 µs, \(q=-0.993962\) and the response remains bounded. At 40.25 µs, \(q=-1.005212\), and its alternating error grows.

The large maximum startup error in native and ME at 50 µs (3.246589 A) is a discretization effect: the step exceeds the 18.1818 µs time constant. Both methods remain bounded and settle; the final error after 5 ms is below \(10^{-7}\,\mathrm{A}\).

## Reproduce

From the repository root, rebuild the fallback FMUs and run the experiment before executing this notebook:

```sh
python3 dpsim-fmi/examples/linear_rl_stability/sources/build_reference_fmus.py
cmake --build build-fmi --target FMI_LinearRLStability -j4
python3 dpsim-fmi/examples/linear_rl_stability/run_benchmark.py
```

The comparison is specific to explicit, non-iterative CS coupling; an iterative rollback/Newton CS policy is a different method and is not tested here.